## The problem

Introduction
You work for a company that makes an online, fantasy-survival game.

When a player finishes a level, they are awarded energy points. The amount of energy awarded depends on which magical items the player found while exploring that level.

Instructions
Your task is to write the code that calculates the energy points that get awarded to players when they complete a level.

The points awarded depend on two things:

The level (a number) that the player completed.
The base value of each magical item collected by the player during that level.
The energy points are awarded according to the following rules:

For each magical item, take the base value and find all the multiples of that value that are less than the level number.
Combine the sets of numbers.
Remove any duplicates.
Calculate the sum of all the numbers that are left.
Let's look at an example:

The player completed level 20 and found two magical items with base values of 3 and 5.

To calculate the energy points earned by the player, we need to find all the unique multiples of these base values that are less than level 20.

Multiples of 3 less than 20: {3, 6, 9, 12, 15, 18}
Multiples of 5 less than 20: {5, 10, 15}
Combine the sets and remove duplicates: {3, 5, 6, 9, 10, 12, 15, 18}
Sum the unique multiples: 3 + 5 + 6 + 9 + 10 + 12 + 15 + 18 = 78
Therefore, the player earns 78 energy points for completing level 20 and finding the two magical items with base values of 3 and 5.
You can make the following assumptions about the inputs to the sum_of_multiples function:

All input numbers are non-negative ints, i.e. natural numbers including zero.
A list of factors must be given, and its elements are unique and sorted in ascending order.


**Summary**

So basically we need to find the sum of multiples for each item in the list up until a certain limit which is the level the player reached in the game. 

### A naive solution

First we have a naive solution. This solution goes through all of the items, and while it's lower than the limit it adds the multiple and repeated additions so 3,6,9... until it's no longer below the limit level. The running time of this algorithm is *O(N)* which means on very large inputs it could take substantial time. For example with a limit of 10000 and five items which all have the base value of 1 it will go through the loop $5 * 10000$ times

In [5]:
def sum_of_multiples_naive(limit, multiples):
    unique_multiples = set()
    for multiple in multiples:
        multiple_counter = multiple
        while multiple < limit:
            unique_multiples.add(multiple)
            multiple = multiple + multiple_counter 
    return sum(unique_multiples)

In [6]:
sum_of_multiples_naive(10, [3, 5])

23

## Rare insights, clever and creative solutions

Instead of brute-forcing our way to the solution we might try to use our mathematical and algorithmic thinking. Consider the multiples of 3 below 30: $3,6,9,12,15,18,21,24,27$. Those numbers correspond to the following $3x1, 3x2, 3x3, 3x4, 3x5, 3x6, 3x7, 3x8, 3x9$. Here we can factor out the $3$ and write $3(1+2+3+4+5+6+7+8+9)$. The sum of this is actually well known to be $k(k+1) /2$ - **Gauss's formula**. 

In this case $k=9$ since Gauss formula is for finding the sum of a sequence from 1 to $k$. This means we have the following sum

$(9*10)/2 = 45$ which we then put back into the original formula so we have $3(45)=135$. 

So now we have a formula for finding the sum of multiples for one item. 

Now let's say we have another item with base value 5. Then the multiples would be $5,10,15,20,25$ and again rewriting plus factoring out we have $5(1,2,3,4,5)$. The sum of this quantity is $5(5*6/2) = 75$

That means in total the sum is $135+75 = 210$. 

But we are not finished yet. If you nottice the multiples of 3 and 5 for this series they both share $15$ and that means right now there are duplicates which there shouldn't be. That means the real sum to return in this case is $210-15 = 195$. Had there been more common multiples they would have to be subtracted to. 

The question is how do we find out which numbers to subtract again? Here we are interested in finding the *least common multiple* of the items. It turns out there is a clever algorithm for finding that which i have worked with earlier. 

This algorithm shown below uses mathematical insights acknowledge by the Greek mathematicians and known as Euclid's algorithm. 

In [39]:
def least_common_multiple(a,b):
    def greatest_common_divisor_euclid(a, b):
        if b == 0:
            return a
        a_prime = a % b
        return greatest_common_divisor_euclid(b, a_prime)

    gcd = greatest_common_divisor_euclid(a,b)
    return int((a*b)/gcd)

When we find the least common multiple between the base values of the items we can eliminate duplicates by subtracting that and all it's multiples lower than the level. For 2,3 and 30 that would only be 15. For 5,10 and 100 that would be 10,20,30,40,50,60,70,80,90. 

In [110]:
import itertools

def least_common_multiple(a,b):
    def greatest_common_divisor_euclid(a, b):
        if b == 0:
            return a
        a_prime = a % b
        return greatest_common_divisor_euclid(b, a_prime)

    gcd = greatest_common_divisor_euclid(a,b)
    return int((a*b)/gcd)

def gauss_sum(lcm, limit):
    k = (limit-1)//lcm
    k = (k*(k+1))//2
    return lcm*k

def chain_lcm(combo):
    result = combo[0]
    for num in range(1, len(combo)):
        result = least_common_multiple(result, combo[num])
    return result

def sum_of_multiples(limit, multiples):
    # initializing sum of multiples
    sum_of_m = 0

    for size in range(1, len(multiples)+1):
        for combo in itertools.combinations(multiples, size):
            if 0 in combo:
                continue
            lcm = chain_lcm(combo)
            total = gauss_sum(lcm,limit)
            if size%2 != 0:
                sum_of_m += total
            else:
                sum_of_m -= total 
            

    return int(sum_of_m)

In [111]:
sum_of_multiples(1, [0])

0

In [77]:
sum_of_multiples(100, [3, 5])

2


2318

In [113]:
sum_of_multiples(10000, [2, 3, 5, 7, 11])

39614537

This solution above now works for the general case and also more efficiently. 

Can we do it more efficient than that? 

Taking a look at some of the community solutions for example this one uses a set generator and a clever use of range to generate the sequence of numbers to sum up. 

In [112]:
def sum_of_multiples(limit, numbers):
    return sum(
        {
            i
            for n in numbers if n
            for i in range(n, limit, n)
        }
    )

## Key takeaways

From this problems one of the key takeways is "When summing a series of evenly spaced numbers, that's Gauss. When combining sets and removing overlaps, that's inclusion-exclusion. When a function works on two inputs and needs to extend to n, check if it chains." 